# 10 — Feature Engineering Fundamentals (Lesson)

## Problem Definition
Construct mathematically grounded tabular feature maps that improve predictive signal without creating leakage.

## Required Prior Knowledge
- Linear algebra basics: vectors, norms, and basis representation.
- Probability basics: expectation and variance.
- Supervised learning setup $(x_i, y_i)$.

## New Concepts Introduced
- Standardization and variance normalization derivation.
- Monotone transforms and log transform behavior.
- One-hot basis expansion.
- Target encoding as conditional expectation $\mathbb{E}[Y\mid C]$.
- Leakage-safe out-of-fold target encoding estimator.
- Bias-variance implications of engineered features.

## Formal Definition
We define a feature map $\phi: \mathbb{R}^d \to \mathbb{R}^p$ and model $f_\theta$ such that
$$
\hat{y}_i = f_\theta(\phi(x_i)).
$$

**Standardization (per feature $j$):**
$$
z_{ij} = \frac{x_{ij}-\mu_j}{\sigma_j},\qquad
\mu_j = \frac{1}{n}\sum_{i=1}^n x_{ij},\qquad
\sigma_j^2 = \frac{1}{n}\sum_{i=1}^n (x_{ij}-\mu_j)^2.
$$
Then
$$
\frac{1}{n}\sum_{i=1}^n z_{ij}=0,\qquad
\frac{1}{n}\sum_{i=1}^n z_{ij}^2=1.
$$

**Gradient scale implication (squared loss, linear model):**
$$
\hat{y}_i = w^\top z_i,\quad
\mathcal{L}(w)=\frac{1}{n}\sum_{i=1}^n(\hat{y}_i-y_i)^2,\quad
\frac{\partial \mathcal{L}}{\partial w_j}
=\frac{2}{n}\sum_{i=1}^n(\hat{y}_i-y_i)z_{ij}.
$$
When $z_{ij}$ are on comparable scales, gradient magnitudes across coordinates are balanced.

**Target encoding (category $c$):**
$$
\operatorname{TE}(c)=\mathbb{E}[Y\mid C=c].
$$
Empirical smoothed estimator:
$$
\widehat{\operatorname{TE}}(c)=
\frac{n_c\bar{y}_c+\alpha\bar{y}}{n_c+\alpha},
$$
where $n_c$ is category count, $\bar{y}_c$ category mean target, $\bar{y}$ global mean, and $\alpha>0$ smoothing.

## Variables and Assumptions
- Scope: Feature Engineering Fundamentals targets the objective `Construct mathematically grounded tabular feature maps that improve predictive signal without creating leakage.`.
- Data are indexed by $i\in\{1,\dots,n\}$ with features $x_i\in\mathbb{R}^d$ and target $y_i$.
- Model parameters are represented by $\theta$, and predictions are $\hat{y}_i=f_\theta(x_i)$.
- Any preprocessing statistics are fit on training partitions only and reused on validation/test partitions.
- Split protocol (KFold/Group/TimeSeries) must match the data-generating assumptions for IID, grouped, or temporal samples.
- Reported metrics are empirical estimates with finite-sample variance; interpretation must include uncertainty.

## Symbol-by-Symbol Explanation
| Symbol | Meaning |
|---|---|
| $x_i$ | feature vector for sample $i$ |
| $y_i$ | target for sample $i$ |
| $f_\theta$ | model parameterized by $\theta$ |
| $L(\cdot,\cdot)$ | per-sample loss function |
| $n$ | number of samples |
| $d$ | raw feature dimension |
| $p$ | transformed feature dimension / polynomial degree context |
| $K$ | number of folds / partitions |
| $V_k$ | validation index set for fold $k$ |
| $\lambda,\alpha,\tau$ | regularization/smoothing/confidence hyperparameters |

## Zero-Skip Derivation
1. Start from raw feature $x_{ij}$ and define centered value:
   $$
   \tilde{x}_{ij}=x_{ij}-\mu_j.
   $$
2. Define normalized value:
   $$
   z_{ij}=\tilde{x}_{ij}/\sigma_j.
   $$
3. Mean check:
   $$
   \frac{1}{n}\sum_i z_{ij}
   =\frac{1}{n\sigma_j}\sum_i(x_{ij}-\mu_j)
   =\frac{1}{n\sigma_j}(n\mu_j-n\mu_j)=0.
   $$
4. Variance check:
   $$
   \frac{1}{n}\sum_i z_{ij}^2
   =\frac{1}{n\sigma_j^2}\sum_i(x_{ij}-\mu_j)^2=1.
   $$
5. For target encoding, split data into folds. For each validation fold, compute category statistics only on training folds:
   $$
   \widehat{\operatorname{TE}}_k(c)
   =\frac{n_{c,-k}\bar{y}_{c,-k}+\alpha\bar{y}_{-k}}{n_{c,-k}+\alpha}.
   $$
   This removes direct access to each sample's own target and prevents leakage.

## Explicit Logical Transitions
0. Context anchor: this notebook focuses on `Feature Engineering Fundamentals` and objective `Construct mathematically grounded tabular feature maps that improve predictive signal without creating leakage.`.
1. Start from the formal objective and identify estimators/transformations introduced in this notebook.
2. Map each estimator term to computable quantities under train/validation split constraints.
3. Show why each derivation step is valid (algebraic identity, estimator definition, or probabilistic assumption).
4. Convert the derivation into an implementation protocol with explicit leakage controls.
5. Validate with synthetic and real-data experiments, then interpret failure conditions.

## Intuition
Standardization improves conditioning, one-hot turns category IDs into basis vectors, and leakage-safe target encoding approximates category signal without peeking at validation targets.

## Mapping from Math to Implementation
- `standardize_train_valid` implements train-only centering/scaling.
- `one_hot_basis` performs basis expansion.
- `target_encode_oof` computes fold-wise conditional expectation estimates.
- Synthetic and real-data sections compare leakage-safe vs leaky encoders.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import torch
import torch.nn as nn

from sklearn.datasets import load_diabetes, load_breast_cancer, load_wine, fetch_california_housing
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, roc_auc_score, accuracy_score
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor, HistGradientBoostingClassifier

try:
    from xgboost import XGBRegressor, XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False

try:
    import optuna
    OPTUNA_AVAILABLE = True
except Exception:
    OPTUNA_AVAILABLE = False

from feature_utils import (
    set_global_seed,
    standardize_train_valid,
    monotone_log1p,
    one_hot_basis,
    polynomial_basis,
    add_interaction_columns,
    target_encode_oof,
    conditional_expectation_feature,
)
from cv_utils import (
    empirical_risk,
    make_splitter,
    oof_cv_predictions,
    cv_bias_variance_decomposition,
    leakage_inflation,
    simulate_public_private_variance,
)
from ensemble_utils import (
    blend_predictions,
    bagging_variance_formula,
    ensemble_variance_from_covariance,
    oof_stacking,
    stacking_predict,
)
from experiment_logger import ExperimentLogger, ExperimentRecord

SEED = 42
set_global_seed(SEED)

MODULE_DIR = Path('.').resolve()

## Synthetic Experiment

In [ ]:
from sklearn.datasets import make_regression

X_num, y = make_regression(n_samples=5000, n_features=4, noise=15.0, random_state=SEED)
rng = np.random.default_rng(SEED)
cats = pd.Series(rng.choice(["A", "B", "C", "D"], size=len(y)))

# Create category-dependent shift in target.
cat_shift = cats.map({"A": 0.0, "B": 20.0, "C": -15.0, "D": 35.0}).to_numpy()
y = y + cat_shift

X_df = pd.DataFrame(X_num, columns=[f"x{i}" for i in range(X_num.shape[1])])
X_df["cat"] = cats
X_df["x0_log"] = monotone_log1p(np.abs(X_df["x0"]))

X_train, X_valid, y_train, y_valid = train_test_split(X_df, y, test_size=0.25, random_state=SEED)
Xtr_std, Xva_std, mu, sigma = standardize_train_valid(X_train[[f"x{i}" for i in range(4)]], X_valid[[f"x{i}" for i in range(4)]])

te_oof_train = target_encode_oof(X_train["cat"], y_train, n_splits=5, smooth=30.0, seed=SEED)
full_stats = pd.DataFrame({"cat": X_train["cat"], "y": y_train}).groupby("cat")["y"].mean()
te_leaky_train = X_train["cat"].map(full_stats).to_numpy(dtype=float)
te_valid = X_valid["cat"].map(full_stats).fillna(np.mean(y_train)).to_numpy(dtype=float)

reg_clean = LinearRegression().fit(np.column_stack([Xtr_std, te_oof_train]), y_train)
reg_leaky = LinearRegression().fit(np.column_stack([Xtr_std, te_leaky_train]), y_train)

pred_clean = reg_clean.predict(np.column_stack([Xva_std, te_valid]))
pred_leaky = reg_leaky.predict(np.column_stack([Xva_std, te_valid]))

rmse_clean = np.sqrt(mean_squared_error(y_valid, pred_clean))
rmse_leaky = np.sqrt(mean_squared_error(y_valid, pred_leaky))
print({"rmse_clean": rmse_clean, "rmse_leaky_train_fit": rmse_leaky})

## Real Dataset Experiment

In [ ]:
diab = load_diabetes(as_frame=True)
X_real = diab.data.copy()
y_real = diab.target.to_numpy(dtype=float)

X_real["bmi_log"] = monotone_log1p(np.maximum(X_real["bmi"], 0.0))
X_tr, X_va, y_tr, y_va = train_test_split(X_real, y_real, test_size=0.25, random_state=SEED)
Xtr_std, Xva_std, _, _ = standardize_train_valid(X_tr.values, X_va.values)

ridge = Ridge(alpha=1.0).fit(Xtr_std, y_tr)
pred = ridge.predict(Xva_std)
rmse = np.sqrt(mean_squared_error(y_va, pred))
print({"diabetes_ridge_rmse": rmse})

## Diagnostic Analysis

In [ ]:
coef = pd.Series(ridge.coef_, index=X_tr.columns).sort_values(key=np.abs, ascending=False)
print(coef.head(8))
plt.figure(figsize=(7, 3))
plt.bar(coef.head(8).index, coef.head(8).values)
plt.xticks(rotation=45)
plt.title("Top coefficient magnitudes (standardized features)")
plt.tight_layout()
plt.show()

## Failure Analysis

In [ ]:
# Failure case: fit scaling statistics on train+valid (data leakage).
full_std = StandardScaler().fit(pd.concat([X_tr, X_va], axis=0))
Xtr_bad = full_std.transform(X_tr)
Xva_bad = full_std.transform(X_va)
ridge_bad = Ridge(alpha=1.0).fit(Xtr_bad, y_tr)
pred_bad = ridge_bad.predict(Xva_bad)
rmse_bad = np.sqrt(mean_squared_error(y_va, pred_bad))
print({"rmse_leaky_scaler": rmse_bad, "rmse_clean_scaler": rmse})

## Exercise Ladder (basic → advanced → research-level)
1. Prove that z-score standardization is invariant to feature translation.
2. Derive the gradient expression for MAE under standardized inputs.
3. Implement James-Stein style shrinkage target encoding and compare with additive smoothing.
4. Show a counterexample where a monotone transform hurts linear separability.

## Summary of Mathematical Insights
Feature engineering is a controlled transformation of representation geometry; mathematically valid transforms plus leakage-safe estimation reduce bias without introducing false validation lift.